<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/Rag_Chtbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 0: Install Gradio in Colab (uncomment if not installed)
# !pip install -q gradio

import csv
import os
import re
import gradio as gr

# =====================================================================
# 1. SMALL PRODUCT KNOWLEDGE BASE
# =====================================================================
KNOWLEDGE_BASE = {
    "cloud server": "Cloud Server Plan costs $25/month with 4 vCPUs, 8GB RAM, and 99.9% uptime.",
    "ai toolkit": "AI Toolkit Pro costs $49/month and includes GPU access and pre-trained APIs.",
    "vpn enterprise": "Enterprise VPN costs $15/month per user with dedicated IPsec tunnels."
}

CSV_FILE = "leads.csv"

# =====================================================================
# 2. PYTHON BACKEND FUNCTION (SAVE LEAD TO CSV)
# =====================================================================
def save_lead_to_csv(name: str, phone: str, product: str, is_hot: bool) -> str:
    """Saves collected lead details into a CSV file."""
    lead_status = "Hot" if is_hot else "Standard"
    file_exists = os.path.isfile(CSV_FILE)

    with open(CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Phone", "Product Interest", "Lead Status"])
        writer.writerow([name, phone, product, lead_status])

    return f"Lead saved! (Name: {name}, Phone: {phone}, Product: {product}, Status: {lead_status})"

# =====================================================================
# 3. CONVERSATIONAL LEAD ASSISTANT LOGIC
# =====================================================================
# Session state to track collected details without guessing
lead_state = {"name": None, "phone": None, "product": None, "buying_intent": False}

def lead_assistant_chat(user_msg, history):
    msg_lower = user_msg.lower()

    # Check clear buying intent
    buy_keywords = ["buy", "purchase", "order", "subscribe", "want to get", "interested in buying"]
    if any(k in msg_lower for k in buy_keywords):
        lead_state["buying_intent"] = True

    # Detect Product Interest from Knowledge Base
    for prod_name, prod_info in KNOWLEDGE_BASE.items():
        if prod_name in msg_lower:
            lead_state["product"] = prod_name.title()
            # If user is just asking a question and not buying yet, answer from KB
            if not lead_state["buying_intent"] and any(q in msg_lower for q in ["price", "cost", "what", "details", "?"]):
                return f"{prod_info}\n\nWould you like to purchase the {prod_name.title()}? If yes, please share your Name and Phone Number."

    # Extract Phone Number (matches digits like 03001234567 or +92...)
    phone_match = re.search(r'(\+?\d[\d\-\s]{7,14}\d)', user_msg)
    if phone_match:
        lead_state["phone"] = phone_match.group(1).strip()

    # Extract Name (if user says "My name is X" or "I am X")
    name_match = re.search(r"(?:name is|i am|i'm|this is)\s+([a-zA-Z\s]{2,25})", user_msg, re.IGNORECASE)
    if name_match:
        lead_state["name"] = name_match.group(1).strip().title()

    # If user wants to buy or has started giving details, check what is missing
    if lead_state["buying_intent"] or lead_state["name"] or lead_state["phone"]:
        missing = []
        if not lead_state["name"]:
            missing.append("your **Name** (e.g., 'My name is Ali')")
        if not lead_state["phone"]:
            missing.append("your **Phone Number**")
        if not lead_state["product"]:
            missing.append("which **Product** you want (Cloud Server, AI Toolkit, or VPN Enterprise)")

        # Never guess missing details -> ask the user
        if missing:
            return f"I'd love to help you with your order! Please provide {', and '.join(missing)}."

        # All details collected -> Call Python function to save in CSV
        confirmation = save_lead_to_csv(
            name=lead_state["name"],
            phone=lead_state["phone"],
            product=lead_state["product"],
            is_hot=lead_state["buying_intent"]
        )
        # Reset state for next customer
        lead_state.update({"name": None, "phone": None, "product": None, "buying_intent": False})
        return f"✅ Thank you! Your request is confirmed.\n{confirmation}"

    # Default greeting / product menu
    return (
        "Hello! Welcome to our store. We offer:\n"
        "1. **Cloud Server** ($25/mo)\n"
        "2. **AI Toolkit** ($49/mo)\n"
        "3. **VPN Enterprise** ($15/mo)\n\n"
        "Ask me about any product, or let me know if you want to buy one!"
    )

# =====================================================================
# 4. GRADIO UI LAUNCHER
# =====================================================================
demo = gr.ChatInterface(
    fn=lead_assistant_chat,
    title="🛍️ AI Lead Assistant (Function Calling Demo)",
    description="Ask about Cloud Server, AI Toolkit, or VPN Enterprise—or say you want to buy!"
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fddd4ed7093bfa2012.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
